# LMNA

Cohort: `LMNA`, Phenopacket Store v0.1.27 (n = 259).

LMNA is the most densely structured cohort in the Phenopacket Store: 1152 term pairs
tested, 205 significant after FDR correction. It is a pleiotropic locus producing
near-disjoint disease axes from a single gene:

- cardiac / skeletal muscle: Emery-Dreifuss muscular dystrophy 2, dilated cardiomyopathy 1A
- metabolic: Dunnigan familial partial lipodystrophy (FPLD2)
- progeroid forms

Correlation results are reported as clusters rather than individual pairs. The
interpretable signal comes from the synergy analysis against disease subtype.

## 1. Load the cohort

In [ ]:
import re
from pathlib import Path

import pandas as pd
from ppktstore.registry import configure_phenopacket_registry

from phenosign import (
    PhenotypeDatasetBuilder,
    HPOCorrelationAnalyzer,
    SynergyAnalyzer,
    has_disease,
)

GENE = "LMNA"
PPKT_RELEASE = "0.1.27"
OUTDIR = Path("../case_studies/LMNA")
OUTDIR.mkdir(parents=True, exist_ok=True)

MIN_INDIVIDUALS = 30
MIN_TARGET_SIZE = 20

In [ ]:
registry = configure_phenopacket_registry()
with registry.open_phenopacket_store(release=PPKT_RELEASE) as ps:
    phenopackets = list(ps.iter_cohort_phenopackets(GENE))

print(f"Loaded {len(phenopackets)} phenopackets")

In [ ]:
# has_disease works on the phenopacket directly, no GPSEA cohort needed here
dataset = PhenotypeDatasetBuilder(phenopackets).build(build_gpsea_cohort=False)

## 2. Disease composition

In [ ]:
diseases_df, sex_df, genes_df, _ = dataset.describe_conditions()
diseases_df.to_csv(OUTDIR / "disease_composition_LMNA.csv", index=False)
diseases_df

## 3. Pairwise correlation

In [ ]:
analyzer = HPOCorrelationAnalyzer(dataset=dataset)
corr_results = analyzer.compute_correlation_matrix()

corr_table = corr_results.results_table
n_sig = int((corr_table["adj_p_value"] < 0.05).sum())
print(f"tested pairs: {len(corr_table)} | significant (adj p < 0.05): {n_sig}")
corr_table.head()

In [ ]:
corr_results.save_correlation_results(
    output_file=str(OUTDIR / "correlation_results_LMNA.csv")
)

In [ ]:
sig = corr_table.query("adj_p_value < 0.05")
sig = sig.reindex(sig["correlation"].abs().sort_values(ascending=False).index)
sig[
    [
        "HPO_A_label",
        "HPO_B_label",
        "correlation",
        "adj_p_value",
        "n_individuals",
        "n_pmids",
    ]
].head(25)

In [ ]:
print("n_pmids across significant pairs:")
print(sig["n_pmids"].describe())
print("\nsingle-publication pairs:", int((sig["n_pmids"] == 1).sum()))

In [ ]:
# threshold raised, the full matrix is unreadable at 205 significant pairs
corr_results.plot_correlation_heatmap_with_significance(
    title_name="Cohort LMNA",
    corr_threshold=0.7,
    adj_pval_threshold=0.05,
)

## 4. Synergy per disease subtype

In [ ]:
print("columns:", list(diseases_df.columns))

id_col = "disease_id" if "disease_id" in diseases_df.columns else diseases_df.columns[0]
label_col = "disease_label" if "disease_label" in diseases_df.columns else diseases_df.columns[1]
count_col = diseases_df.columns[-1]


def parse_count(value):
    # counts come through formatted as '127 (49.0%)'
    if isinstance(value, (int, float)):
        return int(value)
    match = re.match(r"\s*(\d+)", str(value))
    if not match:
        raise ValueError(f"could not parse a count from {value!r}")
    return int(match.group(1))


targets = [
    (row[id_col], row[label_col], parse_count(row[count_col]))
    for _, row in diseases_df.iterrows()
    if parse_count(row[count_col]) >= MIN_TARGET_SIZE
]

for omim_id, label, n in targets:
    print(f"{omim_id}  n={n:>4}  {label}")

In [ ]:
synergy_analyzer = SynergyAnalyzer(
    dataset=dataset,
    min_individuals_for_synergy_calculation=MIN_INDIVIDUALS,
)

synergy_results = {}

for omim_id, label, n in targets:
    slug = omim_id.replace(":", "_")
    print(f"\n=== {omim_id} ({label}), n={n} ===")

    condition = dataset.get_condition(
        has_disease(omim_id),
        name=f"{label} vs other LMNA",
    )
    print(condition.value_counts(dropna=False).to_dict())

    try:
        res = synergy_analyzer.compute_synergy_matrix(condition=condition)
    except ValueError as exc:
        print(f"skipped: {exc}")
        continue

    synergy_results[omim_id] = res
    tbl = res.results_table

    # save_synergy_results() drops everything below synergy >= 0.01,
    # so keep the unfiltered table for the dissynergy side
    tbl.to_csv(OUTDIR / f"synergy_full_LMNA_{slug}.csv", index=False)
    res.save_synergy_results(
        output_file=str(OUTDIR / f"synergy_matrix_LMNA_{slug}.csv")
    )

    print(
        f"rows={len(tbl)} | synergy {tbl['synergy'].min():.4f} to {tbl['synergy'].max():.4f}"
        f" | positive={(tbl['synergy'] > 0).sum()} negative={(tbl['synergy'] < 0).sum()}"
    )

## 5. Synergy heatmaps

In [ ]:
for omim_id, res in synergy_results.items():
    print(f"### {omim_id} ###")
    try:
        fig = res.plot_synergy_heatmap()
    except ValueError as exc:
        print(f"no heatmap: {exc}")
        continue
    if fig is not None:
        display(fig)

## 6. Top synergy and dissynergy per target

The heatmap masks values below `synergy_threshold` and sets `zmin=0`, so the
dissynergy side has to be read from the tables.

In [ ]:
COLS = ["HPO_A_label", "HPO_B_label", "synergy", "adj_p_value", "n_individuals", "n_pmids"]

for omim_id, res in synergy_results.items():
    tbl = res.results_table
    print(f"\n########## {omim_id} ##########")
    print("--- top synergy ---")
    display(tbl.nlargest(10, "synergy")[COLS])
    print("--- top dissynergy ---")
    display(tbl.nsmallest(10, "synergy")[COLS])

In [ ]:
if synergy_results:
    combined = pd.concat(
        [res.results_table.assign(target=omim_id) for omim_id, res in synergy_results.items()],
        ignore_index=True,
    )
    combined.to_csv(OUTDIR / "synergy_all_targets_LMNA.csv", index=False)
    print(combined.groupby("target")["synergy"].describe())

## 7. Variant-effect target

Adds a genotype axis alongside the disease-subtype axis. Needs
`build_gpsea_cohort=True`, which is considerably slower.

In [ ]:
# from phenosign import has_variant_effect
#
# dataset_gpsea = PhenotypeDatasetBuilder(phenopackets).build(build_gpsea_cohort=True)
# condition_lof = dataset_gpsea.get_condition(
#     has_variant_effect("FRAMESHIFT_VARIANT"),
#     name="Truncating vs other LMNA",
# )
# res_lof = SynergyAnalyzer(
#     dataset=dataset_gpsea,
#     min_individuals_for_synergy_calculation=MIN_INDIVIDUALS,
# ).compute_synergy_matrix(condition=condition_lof)
# res_lof.plot_synergy_heatmap()
# res_lof.results_table.to_csv(OUTDIR / "synergy_full_LMNA_truncating.csv", index=False)